# State preparation via alias sampling

Adapted from Qualtran v0.7.0, Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from collections import Counter
from IPython.display import SVG, display
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.circuit import Circuit

## Dense alias preparation

For nonnegative weights $w_l$, let $\lambda=\sum_l w_l$. Coherent alias sampling prepares a state whose selection probabilities approximate $w_l/\lambda$. The corresponding logical junk can remain entangled with selection. The construction controls probabilities. It leaves relative phases unspecified.

Start with a uniform selection word and a uniform $\mu$-bit \(`sigma_mu`\) word. QROM loads the alternate index and the keep numerator. Compare the keep word with the sample, then swap the full selection and alternate words when the keep value is at most the sample. This realizes the classical alias probability of retaining a row, $\mathrm{keep}[l]/2^\mu$.

The through ports are `selection`, `sigma_mu`, `alt`, `keep`, and the comparison bit `less_than_equal`. With selection width $s$, these occupy $2s+2\mu+1$ logical wires. The resource model also accounts for temporary lookup and comparison work. The dense precision argument is an absolute coefficient tolerance, so the table-free model uses $\lambda$ to normalize it.

- [Babbush et al., Encoding electronic spectra in quantum circuits with linear T complexity, Section III.D](https://arxiv.org/abs/1805.03662)

In [ ]:
from examples.algorithms.alias_preparation_catalog import (
    alias_signature,
    dense_alias_example,
    symbolic_alias_example,
)

### Example Instances

In [ ]:
dense = dense_alias_example()
dense.table

In [ ]:
dense_shape = symbolic_alias_example()
dense_shape.resource_model()

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Dense alias", dense.signature, {}),
    ("Symbolic dense alias", alias_signature(dense_shape.selection_width, dense_shape.probability_bits), {}),
])))

### Call Graph

In [ ]:
graph = dense.call_graph()
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
display(estimate_component_resources(dense.resource_model()))

## Sparse alias preparation

For $d$ nonzero coefficients over a domain of size $L$, prepare the uniform state on the support-address word `sparse_index`. The lookup then loads the original selection index, its alternate, and the keep numerator. The comparison and swap operate on the same words as the dense construction. The input may be given as an ordered mapping or a dense list with zeros.

The source port order inserts `sparse_index` between `sigma_mu` and `alt`. Its $\lceil\log_2 d\rceil$ wires accompany the full-domain selection and alternate words, so the externally visible selection width is unchanged. The concrete lookup traverses $d$ rows. In the source table-free sparse factory, the precision model uses the domain size $L$ and normalized tolerance.

- [Berry et al., Qubitization of arbitrary basis quantum chemistry leveraging sparsity and low rank factorization, Section 5](https://arxiv.org/abs/1902.02134)
- [Babbush et al., Section III.D and Figure 11](https://arxiv.org/abs/1805.03662)

In [ ]:
from examples.algorithms.alias_preparation_catalog import sparse_alias_example

### Example Instances

In [ ]:
sparse = sparse_alias_example()
sparse.table

In [ ]:
sparse_from_list = sparse_alias_example(from_dense=True)
assert sparse_from_list.table == sparse.table

In [ ]:
sparse_shape = symbolic_alias_example(sparse=True)
sparse_shape.resource_model()

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Sparse alias from mapping", sparse.signature, {}),
    ("Sparse alias from list", sparse_from_list.signature, {}),
    ("Symbolic sparse alias", alias_signature(
        sparse_shape.selection_width, sparse_shape.probability_bits, sparse_shape.sparse_index_width
    ), {}),
])))

### Call Graph

In [ ]:
graph = sparse.call_graph()
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
display(estimate_component_resources(sparse.resource_model()))

## Emit a circuit

Append each concrete example to a circuit sized for its ports and clean work. The emitted gates and the declarations above use different counting levels.

In [ ]:
for op in (dense, sparse):
    circuit = Circuit(op.qubit_count)
    circuit.append(op)
    display({"reserved wires": op.qubit_count, "emitted gates": Counter(
        gate.name for gate in circuit.to_ir().instructions
    )})